# Cue-Locked EEG Activation EDA

For subjects A1–A5, this notebook separately aggregates every left-hand (`769`) and right-hand (`770`) cue from online Runs 3–6. Each head-shaped topographic heatmap shows the average baseline-normalized 8–30 Hz power during the motor-imagery stimulus interval for all 27 scalp EEG electrodes.

- **0 seconds:** the motor-imagery cue appears.
- **Negative values:** event-related desynchronization (less 8–30 Hz power than baseline).
- **Positive values:** event-related synchronization (more 8–30 Hz power than baseline).

The final intensity at each electrode is averaged across trials of the indicated cue and across the 0-to-5-second stimulus interval. These maps visualize neural activity associated with the stimuli; they are not classifier predictions or per-trial accuracy.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in search_roots
        if (root / "data" / "processed" / "Signals").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the bci_cleaning project")

SIGNALS_ROOT = PROJECT_ROOT / "data" / "processed" / "Signals"
NON_SCALP_CHANNELS = {"EOG1", "EOG2", "EOG3", "EMGg", "EMGd"}
CUE_EVENT_ID = {"Left-hand cue": 769, "Right-hand cue": 770}
FREQUENCY_BAND_HZ = (8.0, 30.0)
TMIN, TMAX = -2.0, 5.0


def subject_sort_key(path):
    match = re.fullmatch(r"([A-Z])(\d+)", path.name)
    return (match.group(1), int(match.group(2))) if match else (path.name, 0)


subject_directories = sorted(
    (
        subject_dir
        for dataset_dir in SIGNALS_ROOT.iterdir()
        if dataset_dir.is_dir()
        for subject_dir in dataset_dir.iterdir()
        if subject_dir.is_dir()
        and re.fullmatch(r"[ABC]\d+", subject_dir.name)
        and list(subject_dir.glob("*_R3_onlineT.gdf"))
    ),
    key=subject_sort_key,
)
selected_subjects = subject_directories[:5]

print("Selected subjects:", [path.name for path in selected_subjects])
for subject_dir in selected_subjects:
    run_files = sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf"))
    print(f"{subject_dir.name}: {len(run_files)} online files")

In [ ]:
def aggregate_subject_activation(subject_dir):
    """Return mean cue-locked 8–30 Hz power change for one participant."""
    run_files = sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf"))
    if not run_files:
        raise FileNotFoundError(f"No online GDF runs found for {subject_dir.name}")

    normalized_trials = {
        "Left-hand cue": [],
        "Right-hand cue": [],
    }
    channel_names = None
    epoch_times = None
    cue_counts = {"Left-hand cue": 0, "Right-hand cue": 0}

    for run_file in run_files:
        raw = mne.io.read_raw_gdf(
            run_file,
            preload=True,
            verbose="ERROR",
        )
        scalp_channels = [
            channel
            for channel in raw.ch_names
            if channel not in NON_SCALP_CHANNELS
        ]
        raw.pick(scalp_channels)

        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(f"Channel order differs in {run_file.name}")

        events, _ = mne.events_from_annotations(
            raw,
            event_id={"769": 769, "770": 770},
            verbose="ERROR",
        )
        if not len(events):
            raw.close()
            continue

        cue_counts["Left-hand cue"] += int((events[:, 2] == 769).sum())
        cue_counts["Right-hand cue"] += int((events[:, 2] == 770).sum())

        # Filter continuously to avoid edge artifacts at individual epoch boundaries.
        raw.filter(
            FREQUENCY_BAND_HZ[0],
            FREQUENCY_BAND_HZ[1],
            method="fir",
            phase="zero",
            verbose="ERROR",
        )
        raw.apply_hilbert(envelope=True, verbose="ERROR")

        epochs = mne.Epochs(
            raw,
            events,
            event_id=CUE_EVENT_ID,
            tmin=TMIN,
            tmax=TMAX,
            baseline=None,
            preload=True,
            decim=8,
            reject_by_annotation=True,
            verbose="ERROR",
        )
        power = np.square(epochs.get_data(copy=False))
        baseline_mask = epochs.times < 0
        baseline_power = power[:, :, baseline_mask].mean(axis=2, keepdims=True)
        baseline_power = np.maximum(baseline_power, np.finfo(float).eps)
        normalized_power = 100 * (power / baseline_power - 1)
        for cue_name, cue_code in CUE_EVENT_ID.items():
            cue_mask = epochs.events[:, 2] == cue_code
            if cue_mask.any():
                normalized_trials[cue_name].append(normalized_power[cue_mask])

        epoch_times = epochs.times.copy()
        raw.close()

    if not all(normalized_trials.values()):
        raise ValueError(f"Both cue types were not found for {subject_dir.name}")

    trials_by_cue = {
        cue_name: np.concatenate(cue_trials, axis=0)
        for cue_name, cue_trials in normalized_trials.items()
    }
    all_trials = np.concatenate(list(trials_by_cue.values()), axis=0)
    return {
        "subject": subject_dir.name,
        "activation": np.nanmean(all_trials, axis=0),
        "activation_by_cue": {
            cue_name: np.nanmean(cue_trials, axis=0)
            for cue_name, cue_trials in trials_by_cue.items()
        },
        "trial_count_by_cue": {
            cue_name: len(cue_trials)
            for cue_name, cue_trials in trials_by_cue.items()
        },
        "times": epoch_times,
        "channels": channel_names,
        "n_trials": len(all_trials),
        "left_cues": cue_counts["Left-hand cue"],
        "right_cues": cue_counts["Right-hand cue"],
        "n_runs": len(run_files),
    }

In [ ]:
subject_activation = []
for subject_dir in selected_subjects:
    print(f"Processing {subject_dir.name}...")
    subject_activation.append(aggregate_subject_activation(subject_dir))

activation_summary = pd.DataFrame(
    {
        "SUJ_ID": result["subject"],
        "online_runs": result["n_runs"],
        "left_cues": result["left_cues"],
        "right_cues": result["right_cues"],
        "aggregated_trials": result["n_trials"],
        "electrodes": len(result["channels"]),
    }
    for result in subject_activation
)
display(activation_summary)

In [ ]:
# Average each cue type and electrode over only the 0-to-5-second stimulus interval.
for result in subject_activation:
    stimulus_mask = (result["times"] >= 0) & (result["times"] <= 5)
    result["stimulus_intensity_by_cue"] = {
        cue_name: np.nanmean(activation[:, stimulus_mask], axis=1)
        for cue_name, activation in result["activation_by_cue"].items()
    }

# Use one robust, symmetric scale so colors are comparable across subjects.
all_intensities = np.concatenate([
    intensity
    for result in subject_activation
    for intensity in result["stimulus_intensity_by_cue"].values()
])
color_limit = np.nanpercentile(np.abs(all_intensities), 98)

montage = mne.channels.make_standard_montage("standard_1020")
topomap_info = mne.create_info(
    subject_activation[0]["channels"],
    sfreq=1.0,
    ch_types="eeg",
)
topomap_info.set_montage(montage, on_missing="raise")

cue_order = ["Left-hand cue", "Right-hand cue"]
fig, axes = plt.subplots(
    len(subject_activation),
    len(cue_order),
    figsize=(12, 24),
    constrained_layout=True,
)

for row, result in enumerate(subject_activation):
    for column, cue_name in enumerate(cue_order):
        ax = axes[row, column]
        image, _ = mne.viz.plot_topomap(
            result["stimulus_intensity_by_cue"][cue_name],
            topomap_info,
            axes=ax,
            show=False,
            sensors="ko",
            names=result["channels"],
            contours=7,
            outlines="head",
            extrapolate="head",
            cmap="RdBu_r",
            vlim=(-color_limit, color_limit),
            res=256,
        )
        ax.set_title(
            f"{result['subject']} — {cue_name}\n"
            f"{result['trial_count_by_cue'][cue_name]} stimulus events",
            fontsize=12,
        )

colorbar = fig.colorbar(image, ax=axes, shrink=0.6, pad=0.02)
colorbar.set_label(
    "Average 8–30 Hz power change during stimulus (%)",
    fontsize=11,
)
fig.suptitle(
    "Left- vs. Right-Hand Cue-Evoked EEG Activation",
    fontsize=17,
)
plt.show()

In [ ]:
# Compare cue activity with 6,900 non-stimulus trial-onset controls.
# Event 768 occurs three seconds before the motor-imagery cue. Its 0-to-2-second
# interval is therefore entirely pre-cue; the preceding two seconds are baseline.
if len(subject_directories) != 87:
    raise ValueError(
        f"Expected 87 participant directories, found {len(subject_directories)}"
    )

CONTROL_LABEL = "Non-stimulus control"
CONTROL_EVENT_CODE = 768
CONTROL_TARGET = 6900
ANALYSIS_TMAX = {"Left-hand cue": 5.0, "Right-hand cue": 5.0,
                 CONTROL_LABEL: 2.0}
base_control_count, extra_controls = divmod(
    CONTROL_TARGET, len(subject_directories)
)
control_target_by_subject = {
    subject_dir.name: base_control_count + (position < extra_controls)
    for position, subject_dir in enumerate(subject_directories)
}


def rms_activation_likelihood(epochs, analysis_tmax):
    """Return one active/not-active value per trial and scalp electrode."""
    baseline_mask = epochs.times < 0
    analysis_mask = (epochs.times >= 0) & (epochs.times <= analysis_tmax)
    voltage_uv = epochs.get_data(copy=False) * 1e6
    baseline_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, baseline_mask]), axis=2
    ))
    analysis_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, analysis_mask]), axis=2
    ))
    return (
        np.isfinite(analysis_rms)
        & np.isfinite(baseline_rms)
        & (analysis_rms > baseline_rms)
    )


def extract_subject_activation_likelihood(subject_dir, control_target):
    """Estimate cue and non-stimulus activation likelihood by electrode."""
    trial_activation = {
        "Left-hand cue": [], "Right-hand cue": [], CONTROL_LABEL: []
    }
    channel_names = None

    for run_file in sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf")):
        raw = mne.io.read_raw_gdf(run_file, preload=True, verbose="ERROR")
        raw.pick([
            channel for channel in raw.ch_names
            if channel not in NON_SCALP_CHANNELS
        ])
        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(f"Channel order differs in {run_file.name}")

        events, _ = mne.events_from_annotations(
            raw,
            event_id={"768": CONTROL_EVENT_CODE, "769": 769, "770": 770},
            verbose="ERROR",
        )
        cue_events = events[np.isin(events[:, 2], [769, 770])]
        control_events = events[events[:, 2] == CONTROL_EVENT_CODE]

        if len(cue_events):
            cue_epochs = mne.Epochs(
                raw, cue_events, event_id=CUE_EVENT_ID,
                tmin=-2.0, tmax=5.0, baseline=(-2.0, 0.0),
                preload=True, reject_by_annotation=True, verbose="ERROR",
            )
            cue_active = rms_activation_likelihood(cue_epochs, 5.0)
            for cue_name, cue_code in CUE_EVENT_ID.items():
                cue_mask = cue_epochs.events[:, 2] == cue_code
                if cue_mask.any():
                    trial_activation[cue_name].append(cue_active[cue_mask])

        if len(control_events):
            control_epochs = mne.Epochs(
                raw, control_events, event_id={CONTROL_LABEL: CONTROL_EVENT_CODE},
                tmin=-2.0, tmax=2.0, baseline=(-2.0, 0.0),
                preload=True, reject_by_annotation=True, verbose="ERROR",
            )
            trial_activation[CONTROL_LABEL].append(
                rms_activation_likelihood(control_epochs, 2.0)
            )
        raw.close()

    trials_by_group = {
        label: np.concatenate(trials, axis=0)
        for label, trials in trial_activation.items()
        if trials
    }
    if set(trials_by_group) != set(trial_activation):
        raise ValueError(f"Missing event group for {subject_dir.name}")
    if len(trials_by_group[CONTROL_LABEL]) < control_target:
        raise ValueError(
            f"Only {len(trials_by_group[CONTROL_LABEL])} controls available for "
            f"{subject_dir.name}; need {control_target}"
        )

    # Evenly spaced deterministic sampling makes the 6,900-event control
    # reproducible and distributes it across each participant's four runs.
    control_indices = np.linspace(
        0, len(trials_by_group[CONTROL_LABEL]) - 1,
        num=control_target, dtype=int,
    )
    trials_by_group[CONTROL_LABEL] = trials_by_group[CONTROL_LABEL][
        control_indices
    ]
    return {
        "subject": subject_dir.name,
        "channels": channel_names,
        "activation_probability": {
            label: 100 * np.nanmean(trials, axis=0)
            for label, trials in trials_by_group.items()
        },
        "event_count": {
            label: len(trials) for label, trials in trials_by_group.items()
        },
    }


all_subject_activation = []
for position, subject_dir in enumerate(subject_directories, start=1):
    print(f"[{position:02d}/87] Processing {subject_dir.name}...")
    all_subject_activation.append(extract_subject_activation_likelihood(
        subject_dir, control_target_by_subject[subject_dir.name]
    ))

# Equal-weight participant averages prevent participants with more retained trials
# from dominating the scalp maps.
group_order = ["Left-hand cue", "Right-hand cue", CONTROL_LABEL]
grand_average_by_group = {
    label: np.nanmean(np.stack([
        result["activation_probability"][label]
        for result in all_subject_activation
    ]), axis=0)
    for label in group_order
}
all_probabilities = np.concatenate(list(grand_average_by_group.values()))
probability_min = np.floor(np.nanmin(all_probabilities))
probability_max = np.ceil(np.nanmax(all_probabilities))
probability_contours = np.linspace(probability_min, probability_max, 8)

grand_montage = mne.channels.make_standard_montage("standard_1020")
grand_topomap_info = mne.create_info(
    all_subject_activation[0]["channels"], sfreq=1.0, ch_types="eeg"
)
grand_topomap_info.set_montage(grand_montage, on_missing="raise")

fig, axes = plt.subplots(1, 3, figsize=(19, 6), constrained_layout=True)
summary_rows = []
for ax, label in zip(axes, group_order):
    probability = grand_average_by_group[label]
    electrode_labels = [
        f"{channel}\n{value:.0f}%"
        for channel, value in zip(
            all_subject_activation[0]["channels"], probability
        )
    ]
    image, _ = mne.viz.plot_topomap(
        probability, grand_topomap_info, axes=ax, show=False,
        sensors="ko", names=electrode_labels,
        contours=probability_contours, outlines="head",
        extrapolate="head", cmap="YlOrRd",
        vlim=(probability_min, probability_max), res=256,
    )
    total_events = sum(
        result["event_count"][label] for result in all_subject_activation
    )
    ax.set_title(
        f"{label}\n87 participants, {total_events:,} events",
        fontsize=13,
    )
    summary_rows.append({
        "event_group": label,
        "participants": len(all_subject_activation),
        "events": total_events,
        "analysis_window_seconds": f"0–{ANALYSIS_TMAX[label]:g}",
    })

colorbar = fig.colorbar(image, ax=axes, shrink=0.8, pad=0.03)
colorbar.set_label(
    "Trials with analysis-window RMS greater than preceding baseline RMS (%)",
    fontsize=11,
)
fig.suptitle(
    "Raw-EEG Electrode Activation Likelihood: Cue vs. Control",
    fontsize=16,
)
plt.show()

grand_average_summary = pd.DataFrame(summary_rows)
display(grand_average_summary)

In [ ]:
# Self-contained analysis: cue and non-stimulus control maps split by gender.
# Dataset coding: SUJ_gender 1 = Man, 2 = Woman.
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

gender_search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
gender_project_root = next(
    (
        root
        for root in gender_search_roots
        if (root / "data" / "processed" / "Signals").is_dir()
    ),
    None,
)
if gender_project_root is None:
    raise RuntimeError("Run this cell from inside the bci_cleaning project")

gender_signals_root = gender_project_root / "data" / "processed" / "Signals"
gender_profile_path = (
    gender_project_root / "data" / "processed" / "Perfomances_cleaned.csv"
)
gender_profile = pd.read_csv(gender_profile_path, sep=";")
gender_lookup = gender_profile.set_index("SUJ_ID")["SUJ_gender"].astype(int)

gender_non_scalp = {"EOG1", "EOG2", "EOG3", "EMGg", "EMGd"}
gender_cue_codes = {"Left-hand cue": 769, "Right-hand cue": 770}
gender_control_label = "Non-stimulus control"
gender_control_code = 768
gender_control_target = 6900


def gender_subject_sort_key(path):
    match = re.fullmatch(r"([A-Z])(\d+)", path.name)
    return (match.group(1), int(match.group(2))) if match else (path.name, 0)


gender_subject_directories = sorted(
    (
        subject_dir
        for dataset_dir in gender_signals_root.iterdir()
        if dataset_dir.is_dir()
        for subject_dir in dataset_dir.iterdir()
        if subject_dir.is_dir()
        and re.fullmatch(r"[ABC]\d+", subject_dir.name)
        and list(subject_dir.glob("*_R3_onlineT.gdf"))
    ),
    key=gender_subject_sort_key,
)
if len(gender_subject_directories) != 87:
    raise ValueError(
        f"Expected 87 participant directories, found "
        f"{len(gender_subject_directories)}"
    )

gender_base_controls, gender_extra_controls = divmod(
    gender_control_target, len(gender_subject_directories)
)
gender_control_target_by_subject = {
    subject_dir.name: gender_base_controls + (position < gender_extra_controls)
    for position, subject_dir in enumerate(gender_subject_directories)
}


def gender_rms_activation_likelihood(epochs, analysis_tmax):
    """Return one active/not-active value per trial and scalp electrode."""
    baseline_mask = epochs.times < 0
    analysis_mask = (epochs.times >= 0) & (epochs.times <= analysis_tmax)
    voltage_uv = epochs.get_data(copy=False) * 1e6
    baseline_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, baseline_mask]), axis=2
    ))
    analysis_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, analysis_mask]), axis=2
    ))
    return (
        np.isfinite(analysis_rms)
        & np.isfinite(baseline_rms)
        & (analysis_rms > baseline_rms)
    )


def extract_gender_cue_activation(subject_dir, control_target):
    """Estimate cue and non-stimulus activation likelihood by electrode."""
    cue_trial_activation = {
        "Left-hand cue": [],
        "Right-hand cue": [],
        gender_control_label: [],
    }
    channel_names = None

    for run_file in sorted(subject_dir.glob("*_R[3-6]_onlineT.gdf")):
        raw = mne.io.read_raw_gdf(
            run_file,
            preload=True,
            verbose="ERROR",
        )
        scalp_channels = [
            channel for channel in raw.ch_names
            if channel not in gender_non_scalp
        ]
        raw.pick(scalp_channels)
        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(f"Channel order differs in {run_file.name}")

        events, _ = mne.events_from_annotations(
            raw,
            event_id={"768": gender_control_code, "769": 769, "770": 770},
            verbose="ERROR",
        )
        cue_events = events[np.isin(events[:, 2], [769, 770])]
        control_events = events[events[:, 2] == gender_control_code]

        if len(cue_events):
            cue_epochs = mne.Epochs(
                raw, cue_events, event_id=gender_cue_codes,
                tmin=-2.0, tmax=5.0, baseline=(-2.0, 0.0),
                preload=True, reject_by_annotation=True, verbose="ERROR",
            )
            cue_is_active = gender_rms_activation_likelihood(cue_epochs, 5.0)
            for cue_name, cue_code in gender_cue_codes.items():
                cue_mask = cue_epochs.events[:, 2] == cue_code
                if cue_mask.any():
                    cue_trial_activation[cue_name].append(
                        cue_is_active[cue_mask]
                    )

        if len(control_events):
            control_epochs = mne.Epochs(
                raw, control_events,
                event_id={gender_control_label: gender_control_code},
                tmin=-2.0, tmax=2.0, baseline=(-2.0, 0.0),
                preload=True, reject_by_annotation=True, verbose="ERROR",
            )
            cue_trial_activation[gender_control_label].append(
                gender_rms_activation_likelihood(control_epochs, 2.0)
            )
        raw.close()

    trials_by_cue = {
        cue_name: np.concatenate(trials, axis=0)
        for cue_name, trials in cue_trial_activation.items()
        if trials
    }
    if set(trials_by_cue) != set(cue_trial_activation):
        raise ValueError(f"Missing event group for {subject_dir.name}")
    if len(trials_by_cue[gender_control_label]) < control_target:
        raise ValueError(
            f"Only {len(trials_by_cue[gender_control_label])} controls available "
            f"for {subject_dir.name}; need {control_target}"
        )
    control_indices = np.linspace(
        0, len(trials_by_cue[gender_control_label]) - 1,
        num=control_target, dtype=int,
    )
    trials_by_cue[gender_control_label] = trials_by_cue[
        gender_control_label
    ][control_indices]
    return {
        "SUJ_ID": subject_dir.name,
        "SUJ_gender": int(gender_lookup.loc[subject_dir.name]),
        "channels": channel_names,
        "activation_probability_by_cue": {
            cue_name: 100 * np.nanmean(trials, axis=0)
            for cue_name, trials in trials_by_cue.items()
        },
        "trial_count_by_cue": {
            cue_name: len(trials)
            for cue_name, trials in trials_by_cue.items()
        },
    }


gender_subject_results = []
for position, subject_dir in enumerate(gender_subject_directories, start=1):
    print(f"[{position:02d}/87] Processing {subject_dir.name}...")
    gender_subject_results.append(extract_gender_cue_activation(
        subject_dir, gender_control_target_by_subject[subject_dir.name]
    ))

gender_codes = [1, 2]
gender_labels = {1: "Gender 1 (Man)", 2: "Gender 2 (Woman)"}
gender_cue_order = [
    "Left-hand cue", "Right-hand cue", gender_control_label
]
gender_grand_average = {}
for gender_code in gender_codes:
    gender_members = [
        result
        for result in gender_subject_results
        if result["SUJ_gender"] == gender_code
    ]
    for cue_name in gender_cue_order:
        gender_grand_average[(gender_code, cue_name)] = np.nanmean(
            np.stack([
                result["activation_probability_by_cue"][cue_name]
                for result in gender_members
            ]),
            axis=0,
        )

gender_all_probabilities = np.concatenate(
    list(gender_grand_average.values())
)
gender_probability_min = np.floor(np.nanmin(gender_all_probabilities))
gender_probability_max = np.ceil(np.nanmax(gender_all_probabilities))
gender_probability_contours = np.linspace(
    gender_probability_min, gender_probability_max, 8
)

gender_montage = mne.channels.make_standard_montage("standard_1020")
gender_topomap_info = mne.create_info(
    gender_subject_results[0]["channels"],
    sfreq=1.0,
    ch_types="eeg",
)
gender_topomap_info.set_montage(gender_montage, on_missing="raise")

fig, axes = plt.subplots(2, 3, figsize=(21, 13), constrained_layout=True)
gender_summary_rows = []
for row, gender_code in enumerate(gender_codes):
    members = [
        result
        for result in gender_subject_results
        if result["SUJ_gender"] == gender_code
    ]
    for column, cue_name in enumerate(gender_cue_order):
        ax = axes[row, column]
        activation_probability = gender_grand_average[
            (gender_code, cue_name)
        ]
        electrode_labels = [
            f"{channel}\n{probability:.0f}%"
            for channel, probability in zip(
                gender_subject_results[0]["channels"],
                activation_probability,
            )
        ]
        image, _ = mne.viz.plot_topomap(
            activation_probability,
            gender_topomap_info,
            axes=ax,
            show=False,
            sensors="ko",
            names=electrode_labels,
            contours=gender_probability_contours,
            outlines="head",
            extrapolate="head",
            cmap="YlOrRd",
            vlim=(gender_probability_min, gender_probability_max),
            res=256,
        )
        total_trials = sum(
            result["trial_count_by_cue"][cue_name]
            for result in members
        )
        ax.set_title(
            f"{gender_labels[gender_code]} — {cue_name}\n"
            f"{len(members)} participants, {total_trials:,} events",
            fontsize=12,
        )
        gender_summary_rows.append(
            {
                "gender": gender_labels[gender_code],
                "cue": cue_name,
                "participants": len(members),
                "events": total_trials,
                "analysis_window_seconds": (
                    "0–2" if cue_name == gender_control_label else "0–5"
                ),
            }
        )

colorbar = fig.colorbar(image, ax=axes, shrink=0.7, pad=0.03)
colorbar.set_label(
    "Trials with analysis-window RMS greater than preceding baseline RMS (%)",
    fontsize=11,
)
fig.suptitle(
    "Raw-EEG Electrode Activation Likelihood by Gender: Cue vs. Control",
    fontsize=16,
)
plt.show()

gender_grand_average_summary = pd.DataFrame(gender_summary_rows)
display(gender_grand_average_summary)

In [ ]:
# Aggregate the dedicated closed-eyes and open-eyes baseline GDF recordings.
# Each five-second analysis window is compared with its preceding two seconds,
# matching the RMS activation-likelihood metric used in cells 6 and 7.
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

baseline_search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
baseline_project_root = next(
    (
        root
        for root in baseline_search_roots
        if (root / "data" / "processed" / "Signals").is_dir()
    ),
    None,
)
if baseline_project_root is None:
    raise RuntimeError("Run this cell from inside the bci_cleaning project")

baseline_signals_root = (
    baseline_project_root / "data" / "processed" / "Signals"
)
baseline_non_scalp = {"EOG1", "EOG2", "EOG3", "EMGg", "EMGd"}
baseline_conditions = {
    "Closed eyes": "*_CE_baseline.gdf",
    "Open eyes": "*_OE_baseline.gdf",
}
baseline_window_target = 6900


def baseline_subject_sort_key(path):
    match = re.fullmatch(r"([A-Z])(\d+)", path.name)
    return (match.group(1), int(match.group(2))) if match else (path.name, 0)


baseline_subject_directories = sorted(
    (
        subject_dir
        for dataset_dir in baseline_signals_root.iterdir()
        if dataset_dir.is_dir()
        for subject_dir in dataset_dir.iterdir()
        if subject_dir.is_dir()
        and re.fullmatch(r"[ABC]\d+", subject_dir.name)
        and all(list(subject_dir.glob(pattern)) for pattern in baseline_conditions.values())
    ),
    key=baseline_subject_sort_key,
)
if len(baseline_subject_directories) != 87:
    raise ValueError(
        f"Expected 87 participants with both baseline files, found "
        f"{len(baseline_subject_directories)}"
    )

# Allocate exactly 6,900 windows across participants, then split each quota
# as evenly as possible between closed-eyes and open-eyes recordings.
baseline_base_count, baseline_extra_count = divmod(
    baseline_window_target, len(baseline_subject_directories)
)
baseline_targets = {}
for position, subject_dir in enumerate(baseline_subject_directories):
    subject_target = baseline_base_count + (position < baseline_extra_count)
    closed_target = subject_target // 2
    if subject_target % 2 and position % 2 == 0:
        closed_target += 1
    baseline_targets[subject_dir.name] = {
        "Closed eyes": closed_target,
        "Open eyes": subject_target - closed_target,
    }


def baseline_rms_activation_likelihood(epochs):
    """Return one active/not-active value per window and scalp electrode."""
    baseline_mask = epochs.times < 0
    analysis_mask = (epochs.times >= 0) & (epochs.times <= 5)
    voltage_uv = epochs.get_data(copy=False) * 1e6
    baseline_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, baseline_mask]), axis=2
    ))
    analysis_rms = np.sqrt(np.nanmean(
        np.square(voltage_uv[:, :, analysis_mask]), axis=2
    ))
    return (
        np.isfinite(analysis_rms)
        & np.isfinite(baseline_rms)
        & (analysis_rms > baseline_rms)
    )


def extract_baseline_activation(subject_dir, condition_targets):
    """Aggregate sampled baseline windows for one participant."""
    selected_windows = []
    selected_count_by_condition = {}
    channel_names = None

    for condition, pattern in baseline_conditions.items():
        matching_files = sorted(subject_dir.glob(pattern))
        if len(matching_files) != 1:
            raise ValueError(
                f"Expected one {condition} file for {subject_dir.name}, found "
                f"{len(matching_files)}"
            )
        raw = mne.io.read_raw_gdf(
            matching_files[0], preload=True, verbose="ERROR"
        )
        raw.pick([
            channel for channel in raw.ch_names
            if channel not in baseline_non_scalp
        ])
        if channel_names is None:
            channel_names = raw.ch_names.copy()
        elif raw.ch_names != channel_names:
            raise ValueError(
                f"Channel order differs in {matching_files[0].name}"
            )

        recording_starts = [
            float(annotation["onset"])
            for annotation in raw.annotations
            if str(annotation["description"]) == "32775"
        ]
        recording_ends = [
            float(annotation["onset"])
            for annotation in raw.annotations
            if str(annotation["description"]) == "32776"
        ]
        if not recording_starts:
            raise ValueError(
                f"No baseline start marker in {matching_files[0].name}"
            )

        # Some recordings contain near-duplicate start/end annotations (A16 and
        # A32), while A30 has no end annotation. Collapse duplicates to the widest
        # marked interval and fall back to the recording boundary when needed.
        recording_start = min(recording_starts)
        recording_end = (
            max(recording_ends) if recording_ends else float(raw.times[-1])
        )
        if recording_end - recording_start < 7.0:
            raise ValueError(
                f"Baseline interval in {matching_files[0].name} is too short: "
                f"{recording_end - recording_start:.1f} seconds"
            )

        # Two-second spacing supplies a reproducible pool across the continuous
        # baseline while keeping every full -2-to-5-second epoch in that interval.
        anchor_times = np.arange(
            recording_start + 2.0,
            recording_end - 5.0 + np.finfo(float).eps,
            2.0,
        )
        synthetic_events = np.column_stack([
            raw.time_as_index(anchor_times) + raw.first_samp,
            np.zeros(len(anchor_times), dtype=int),
            np.full(len(anchor_times), 999, dtype=int),
        ])
        epochs = mne.Epochs(
            raw, synthetic_events, event_id={"Baseline window": 999},
            tmin=-2.0, tmax=5.0, baseline=(-2.0, 0.0),
            preload=True, reject_by_annotation=True, verbose="ERROR",
        )
        available_windows = baseline_rms_activation_likelihood(epochs)
        target = condition_targets[condition]
        if len(available_windows) < target:
            raise ValueError(
                f"Only {len(available_windows)} {condition} windows available "
                f"for {subject_dir.name}; need {target}"
            )
        selected_indices = np.linspace(
            0, len(available_windows) - 1, num=target, dtype=int
        )
        selected_windows.append(available_windows[selected_indices])
        selected_count_by_condition[condition] = target
        raw.close()

    combined_windows = np.concatenate(selected_windows, axis=0)
    return {
        "subject": subject_dir.name,
        "channels": channel_names,
        "activation_probability": 100 * np.nanmean(combined_windows, axis=0),
        "window_count": len(combined_windows),
        "window_count_by_condition": selected_count_by_condition,
    }


baseline_subject_results = []
for position, subject_dir in enumerate(baseline_subject_directories, start=1):
    print(f"[{position:02d}/87] Processing {subject_dir.name} baselines...")
    baseline_subject_results.append(extract_baseline_activation(
        subject_dir, baseline_targets[subject_dir.name]
    ))

# Give every participant equal weight in the final electrode map.
baseline_grand_average = np.nanmean(np.stack([
    result["activation_probability"] for result in baseline_subject_results
]), axis=0)
baseline_probability_min = np.floor(np.nanmin(baseline_grand_average))
baseline_probability_max = np.ceil(np.nanmax(baseline_grand_average))
baseline_contours = np.linspace(
    baseline_probability_min, baseline_probability_max, 8
)

baseline_montage = mne.channels.make_standard_montage("standard_1020")
baseline_topomap_info = mne.create_info(
    baseline_subject_results[0]["channels"], sfreq=1.0, ch_types="eeg"
)
baseline_topomap_info.set_montage(baseline_montage, on_missing="raise")
baseline_electrode_labels = [
    f"{channel}\n{probability:.0f}%"
    for channel, probability in zip(
        baseline_subject_results[0]["channels"], baseline_grand_average
    )
]

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
image, _ = mne.viz.plot_topomap(
    baseline_grand_average, baseline_topomap_info, axes=ax, show=False,
    sensors="ko", names=baseline_electrode_labels,
    contours=baseline_contours, outlines="head", extrapolate="head",
    cmap="YlOrRd",
    vlim=(baseline_probability_min, baseline_probability_max), res=256,
)
total_baseline_windows = sum(
    result["window_count"] for result in baseline_subject_results
)
ax.set_title(
    "Closed- and Open-Eyes Baseline EEG\n"
    f"87 participants, {total_baseline_windows:,} sampled windows",
    fontsize=14,
)
colorbar = fig.colorbar(image, ax=ax, shrink=0.8, pad=0.03)
colorbar.set_label(
    "Windows with analysis RMS greater than preceding baseline RMS (%)",
    fontsize=11,
)
plt.show()

baseline_summary = pd.DataFrame([{
    "participants": len(baseline_subject_results),
    "baseline_files": 2 * len(baseline_subject_results),
    "closed_eyes_windows": sum(
        result["window_count_by_condition"]["Closed eyes"]
        for result in baseline_subject_results
    ),
    "open_eyes_windows": sum(
        result["window_count_by_condition"]["Open eyes"]
        for result in baseline_subject_results
    ),
    "total_windows": total_baseline_windows,
    "analysis_window_seconds": "0–5",
}])
display(baseline_summary)